# Train Notebook

- epoch당 10000개 train sample을 랜덤 시퀀스로 생성
- valid loss, note count, note precision/recall 출력
- best-valid 모델과 주기별 후보 checkpoint 저장
- `RUN_MODE='base'`: 전체 mood 기반 base 학습
- `RUN_MODE='finetune'`: 특정 mood/style fine-tuning


## 0. 기본 설정
필요한 라이브러리를 불러오고 Google Drive 마운트, 실행 디바이스, 난수 seed를 설정한다.


In [ ]:
import os
import random
import csv
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch import optim
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ 사용 중인 디바이스: {device}")

# 학습 재현성을 위해 Python/Torch 난수 시드를 고정한다.
def seed_everything(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)


## 1. 학습 데이터셋 구성
mood별 pt 파일을 train/valid로 나누고, 매 epoch마다 랜덤 연속 chunk 시퀀스를 샘플링한다.


In [ ]:
# mood별 .pt 청크 파일에서 연속 시퀀스를 샘플링하는 Dataset이다.
class SequenceDataset(Dataset):
    # 학습/검증 split을 파일 단위로 나누고, 사용할 mood와 genre 파일 목록을 준비한다.
    def __init__(
        self,
        data_root,
        refresh_rate=1000,
        seq_len=8,
        split='train',
        val_ratio=0.1,
        max_files_per_mood=247,
        samples_per_epoch=10000,
        seed=42,
        target_moods=None,
    ):
        self.data_root = data_root
        self.refresh_rate = refresh_rate
        self.seq_len = seq_len
        self.split = split
        self.samples_per_epoch = samples_per_epoch
        self.seed = seed

        self.mood_map = {
            'angry': 0, 'exciting': 1, 'fear': 2, 'funny': 3, 'happy': 4,
            'lazy': 5, 'magnificent': 6, 'quiet': 7, 'romantic': 8, 'sad': 9, 'warm': 10
        }

        # 각 mood에서 대표로 사용할 genre 폴더를 지정한다.
        self.mood_genre_map = {
            'angry': 'rock',
            'exciting': 'pop',
            'fear': 'rock',
            'funny': 'pop',
            'happy': 'country',
            'lazy': 'jazz',
            'magnificent': 'classical',
            'quiet': 'classical',
            'romantic': 'country',
            'sad': 'classical',
            'warm': 'country'
        }

        if split not in ['train', 'valid']:
            raise ValueError("split must be either 'train' or 'valid'")

        if target_moods is None:
            self.active_moods = list(self.mood_map.keys())
        else:
            self.active_moods = list(target_moods)
            for mood in self.active_moods:
                if mood not in self.mood_map:
                    raise ValueError(f"Unknown mood: {mood}")

        self.mood_files = {}
        self.mood_queues = {}
        self.active_data = {}
        self.active_file = {}
        self.call_counts = {mood: 0 for mood in self.active_moods}

        print(f"\n📁 Dataset split = {split}")
        print(f"🎯 Active moods = {self.active_moods}")
        print("-" * 70)

        for mood in self.active_moods:
            genre = self.mood_genre_map[mood]
            folder = os.path.join(data_root, mood, genre)

            if not os.path.exists(folder):
                self.mood_files[mood] = []
                self.mood_queues[mood] = []
                print(f"⚠️ 폴더 없음: {folder}")
                continue

            files = sorted([
                os.path.join(folder, f)
                for f in os.listdir(folder)
                if f.endswith('.pt')
            ])

            original_count = len(files)

            # mood별 파일 수를 동일하게 맞춰 데이터 불균형을 줄인다.
            if max_files_per_mood is not None:
                files = files[:max_files_per_mood]

            capped_count = len(files)
            if capped_count < 2:
                raise RuntimeError(f"{mood}/{genre} has too few files after cap: {capped_count}")

            # 같은 mood/genre 안에서 train과 valid 파일을 고정된 seed로 분리한다.
            rng = random.Random(seed + self.mood_map[mood])
            rng.shuffle(files)

            val_count = max(1, int(capped_count * val_ratio))
            if split == 'valid':
                selected_files = files[:val_count]
            else:
                selected_files = files[val_count:]

            if len(selected_files) == 0:
                raise RuntimeError(f"{split} selected 0 files for {mood}/{genre}. Increase max_files_per_mood.")

            self.mood_files[mood] = selected_files
            self.mood_queues[mood] = []

            print(
                f"{mood}/{genre}: original={original_count}, capped={capped_count}, "
                f"{split}={len(selected_files)} files"
            )

        print("-" * 70)
        print(
            f"⏳ active 파일만 메모리에 로드합니다. "
            f"전체 후보 파일을 전부 torch.load 하는 구조가 아닙니다."
        )
        for mood in self.active_moods:
            self._load_file(mood)
        print(f"✅ {split} dataset 준비 완료!\n")

    # 특정 mood의 다음 .pt 파일을 메모리에 올리고 샘플링 카운터를 초기화한다.
    def _load_file(self, mood):
        if not self.mood_files[mood]:
            return

        if len(self.mood_queues[mood]) == 0:
            self.mood_queues[mood] = self.mood_files[mood].copy()
            random.shuffle(self.mood_queues[mood])

        file_path = self.mood_queues[mood].pop()
        data = torch.load(file_path, map_location='cpu')
        self.active_data[mood] = data['chunks']
        self.active_file[mood] = file_path
        self.call_counts[mood] = 0

    # DataLoader가 한 epoch에서 사용할 샘플 개수를 반환한다.
    def __len__(self):
        return self.samples_per_epoch

    # 랜덤 mood와 랜덤 시작 위치를 골라 입력 시퀀스와 다음 청크 타깃을 만든다.
    def __getitem__(self, idx):
        while True:
            mood = random.choice(self.active_moods)

            # 같은 파일을 너무 오래 쓰지 않도록 refresh_rate마다 파일을 교체한다.
            if self.call_counts[mood] >= self.refresh_rate:
                self._load_file(mood)

            chunks = self.active_data.get(mood, [])
            max_idx = len(chunks) - self.seq_len - 1

            if max_idx <= 0:
                self._load_file(mood)
                continue

            # 한 곡 안에서 seq_len개 입력과 그 다음 seq_len개 타깃을 만든다.
            c_idx = random.randint(0, max_idx)
            seq_chunks = chunks[c_idx : c_idx + self.seq_len + 1]
            self.call_counts[mood] += 1

            x_seq = seq_chunks[:-1].float()
            y_seq = seq_chunks[1:].float()
            mood_tensor = torch.tensor(self.mood_map[mood], dtype=torch.long)

            return x_seq, y_seq, mood_tensor


## 2. CVAE-GRU 모델 정의
CNN encoder/decoder와 GRU hidden state를 사용해 mood 조건 기반 다음 piano-roll chunk를 예측한다.


In [ ]:
# mood 조건을 함께 사용하는 CNN-VAE + GRU 기반 다음 청크 예측 모델이다.
class CVAE_GRU(nn.Module):
    # 인코더, GRU 상태 갱신부, 잠재변수 층, 디코더를 정의한다.
    def __init__(self, num_moods=11, embed_dim=32, channels=4, feature_dim=256, hidden_dim=256, latent_dim=128):
        super(CVAE_GRU, self).__init__()
        self.hidden_dim = hidden_dim
        self.mood_embedding = nn.Embedding(num_moods, embed_dim)

        self.encoder_cnn = nn.Sequential(
            nn.Conv2d(channels, 32, kernel_size=3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), nn.ReLU(),
            nn.Flatten()
        )

        self.cnn_shape = (128, 11, 4)
        self.flatten_dim = self.cnn_shape[0] * self.cnn_shape[1] * self.cnn_shape[2]
        self.feature_proj = nn.Linear(self.flatten_dim, feature_dim)

        self.gru_cell = nn.GRUCell(input_size=feature_dim + embed_dim, hidden_size=hidden_dim)

        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        self.decoder_proj = nn.Linear(latent_dim + embed_dim + hidden_dim, self.flatten_dim)

        self.decoder_cnn = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=(1,1)), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=(1,1)), nn.ReLU(),
            nn.ConvTranspose2d(32, channels, kernel_size=3, stride=2, padding=1, output_padding=(1,1)),
            nn.Sigmoid()
        )

    # 입력 청크와 mood를 feature로 바꾸고 GRU hidden state에서 mu/logvar를 계산한다.
    def encode(self, x, mood_vector, prev_hidden):
        cnn_features = self.encoder_cnn(x)
        features = F.relu(self.feature_proj(cnn_features))
        gru_input = torch.cat([features, mood_vector], dim=1)
        current_hidden = self.gru_cell(gru_input, prev_hidden)

        mu = self.fc_mu(current_hidden)
        logvar = self.fc_logvar(current_hidden)
        return mu, logvar, current_hidden

    # VAE 학습을 위해 mu/logvar에서 미분 가능한 방식으로 z를 샘플링한다.
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    # 잠재변수, mood, 현재 hidden state를 다음 청크 형태로 복원한다.
    def decode(self, z, mood_vector, current_hidden):
        decoder_input = torch.cat([z, mood_vector, current_hidden], dim=1)
        x = F.relu(self.decoder_proj(decoder_input))
        x = x.view(-1, *self.cnn_shape)
        return self.decoder_cnn(x)

    # 한 시점의 입력 청크를 받아 복원 결과와 VAE 파라미터, 다음 hidden state를 반환한다.
    def forward(self, x, mood_id, prev_hidden=None):
        batch_size = x.size(0)
        if prev_hidden is None:
            prev_hidden = torch.zeros(batch_size, self.hidden_dim).to(x.device)

        mood_vector = self.mood_embedding(mood_id)
        mu, logvar, current_hidden = self.encode(x, mood_vector, prev_hidden)
        z = self.reparameterize(mu, logvar)
        reconstruction = self.decode(z, mood_vector, current_hidden)

        return reconstruction, mu, logvar, current_hidden


## 3. 학습 함수와 평가 지표
VAE loss, note precision/recall, early stopping, epoch 실행 함수를 정의한다.


In [ ]:
# BCE 재구성 손실과 KL divergence를 합쳐 VAE loss를 계산한다.
def vae_loss_fn(recon_x, x, mu, logvar, beta=1.0):
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    total_loss = (BCE + beta * KLD) / x.size(0)
    return total_loss, BCE / x.size(0), KLD / x.size(0)


# note 예측 품질을 누적할 통계 딕셔너리를 초기화한다.
def init_note_stats():
    return {
        'pred_notes': 0.0,
        'target_notes': 0.0,
        'tp': 0.0,
        'fp': 0.0,
        'fn': 0.0,
        'chunks': 0,
    }


# threshold 기준으로 예측 note와 실제 note의 precision/recall 계산 재료를 누적한다.
def update_note_stats(stats, recon, target, threshold=0.5):
    with torch.no_grad():
        pred = recon.detach() >= threshold
        tgt = target.detach() >= threshold

        stats['pred_notes'] += pred.float().sum().item()
        stats['target_notes'] += tgt.float().sum().item()
        stats['tp'] += (pred & tgt).float().sum().item()
        stats['fp'] += (pred & (~tgt)).float().sum().item()
        stats['fn'] += ((~pred) & tgt).float().sum().item()
        stats['chunks'] += recon.size(0)


# 누적된 note 통계를 사람이 보기 쉬운 metric 값으로 변환한다.
def finalize_note_stats(stats):
    eps = 1e-8
    precision = stats['tp'] / (stats['tp'] + stats['fp'] + eps)
    recall = stats['tp'] / (stats['tp'] + stats['fn'] + eps)
    pred_notes_per_chunk = stats['pred_notes'] / max(1, stats['chunks'])
    target_notes_per_chunk = stats['target_notes'] / max(1, stats['chunks'])
    return {
        'pred_notes_per_chunk': pred_notes_per_chunk,
        'target_notes_per_chunk': target_notes_per_chunk,
        'note_precision': precision,
        'note_recall': recall,
    }


# valid loss가 개선될 때 best checkpoint를 저장하고, 오래 개선되지 않으면 중단 신호를 만든다.
class EarlyStopping:
    # patience, 최소 학습 epoch, 저장 경로 등 early stopping 설정을 초기화한다.
    def __init__(self, patience=30, min_epochs=30, min_delta=0.0, save_path='best_valid_model.pth', enabled=True):
        self.patience = patience
        self.min_epochs = min_epochs
        self.min_delta = min_delta
        self.save_path = save_path
        self.enabled = enabled
        self.counter = 0
        self.best_loss = None
        self.best_epoch = None
        self.early_stop = False
        os.makedirs(os.path.dirname(self.save_path), exist_ok=True)

    # 현재 valid loss를 best loss와 비교해 checkpoint 저장 또는 중단 여부를 갱신한다.
    def __call__(self, val_loss, model, optimizer, epoch, extra=None):
        improved = self.best_loss is None or val_loss < (self.best_loss - self.min_delta)

        if improved:
            self.best_loss = val_loss
            self.best_epoch = epoch
            self.counter = 0
            payload = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'valid_loss': val_loss,
            }
            if extra is not None:
                payload.update(extra)
            torch.save(payload, self.save_path)
            print(f"💾 best-valid 저장: epoch {epoch+1}, valid loss {val_loss:.4f}")
        else:
            self.counter += 1

        if self.enabled and (epoch + 1) >= self.min_epochs and self.counter >= self.patience:
            self.early_stop = True


# train 또는 valid loader를 한 epoch 실행하고 평균 loss/metric을 반환한다.
def run_epoch(model, loader, optimizer=None, beta_val=1.0, note_threshold=0.5, track_note=True):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_bce = 0.0
    total_kld = 0.0
    note_stats = init_note_stats()

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for x_seq, y_seq, mood_ids in loader:
            x_seq = x_seq.to(device, non_blocking=True)
            y_seq = y_seq.to(device, non_blocking=True)
            mood_ids = mood_ids.to(device, non_blocking=True)

            if is_train:
                optimizer.zero_grad()

            batch_loss = 0.0
            batch_bce = 0.0
            batch_kld = 0.0
            hidden = None
            seq_len = x_seq.size(1)

            for t in range(seq_len):
                x_t = x_seq[:, t, :, :, :]
                y_t = y_seq[:, t, :, :, :]

                recon, mu, logvar, hidden = model(x_t, mood_ids, prev_hidden=hidden)
                loss, bce, kld = vae_loss_fn(recon, y_t, mu, logvar, beta=beta_val)

                batch_loss = batch_loss + loss
                batch_bce = batch_bce + bce
                batch_kld = batch_kld + kld

                if track_note:
                    update_note_stats(note_stats, recon, y_t, threshold=note_threshold)

            if is_train:
                batch_loss.backward()
                optimizer.step()

            total_loss += (batch_loss.item() / seq_len)
            total_bce += (batch_bce.item() / seq_len)
            total_kld += (batch_kld.item() / seq_len)

    n = len(loader)
    metrics = {
        'loss': total_loss / n,
        'bce': total_bce / n,
        'kld': total_kld / n,
    }
    if track_note:
        metrics.update(finalize_note_stats(note_stats))
    return metrics


## 4. 학습 실행과 checkpoint 저장
경로와 하이퍼파라미터를 설정한 뒤 train/valid loop를 돌리고 best/latest/candidate checkpoint와 loss 그래프를 저장한다.


In [ ]:
# 학습 데이터, 저장 위치, 하이퍼파라미터 설정
DATA_ROOT = '/content/drive/MyDrive/midivae/data'
SAVE_ROOT = '/content/drive/MyDrive/midivae/saved_model/final_train'
FIG_ROOT = '/content/drive/MyDrive/midivae/loss_figure/final_train'

# base는 전체 mood를 학습하고, finetune은 지정한 mood만 추가 학습한다.
RUN_MODE = 'base'  # 'base' or 'finetune'
TARGET_MOODS = ['sad']  # finetune에서만 사용

SEQ_LEN = 8
BATCH_SIZE = 64
REFRESH_RATE = 1000
MAX_FILES_PER_MOOD = 247
VAL_RATIO = 0.1

# 한 epoch에서 Dataset이 제공할 가상 샘플 수
TRAIN_SAMPLES_PER_EPOCH = 10000
VALID_SAMPLES_PER_EPOCH = 2200

EPOCHS = 247
LR = 1e-4
BETA_ANNEAL_RATIO = 0.5
NOTE_THRESHOLD = 0.5

# base 학습은 valid loss 기반 early stopping을 켤 수 있고, finetune은 checkpoint 비교를 우선한다.
USE_EARLY_STOPPING = False  # True if RUN_MODE == 'base' else False
MIN_EPOCHS_BEFORE_STOP = 30
PATIENCE = 30

# 생성 결과를 비교할 후보 checkpoint 저장 주기
CHECKPOINT_EVERY = 10
EXTRA_SAVE_EPOCHS = {5, 15, 20, 30, 50, 80, 100, 150, 200, 247}

NUM_WORKERS = 2
PIN_MEMORY = True

# finetune을 base best-valid 모델에서 시작할지 설정
LOAD_BASE_FOR_FINETUNE = True
BASE_BEST_PATH = os.path.join(SAVE_ROOT, 'base_topgenre247', 'best_valid_model.pth')

# 실행 모드에 맞는 저장 경로 생성
if RUN_MODE == 'base':
    active_moods = None
    run_name = 'base_topgenre247'
else:
    active_moods = TARGET_MOODS
    run_name = 'finetune_' + '_'.join(TARGET_MOODS)

RUN_DIR = os.path.join(SAVE_ROOT, run_name)
CKPT_DIR = os.path.join(RUN_DIR, 'checkpoints')
FIG_DIR = os.path.join(FIG_ROOT, run_name)
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

BEST_VALID_MODEL_PATH = os.path.join(RUN_DIR, 'best_valid_model.pth')
LATEST_CHECKPOINT_PATH = os.path.join(RUN_DIR, 'latest_checkpoint.pth')
LOSS_LOG_PATH = os.path.join(FIG_DIR, 'loss_log.csv')
GRAPH_PATH = os.path.join(FIG_DIR, 'loss_graph.png')

print("=" * 80)
print(f"RUN_MODE: {RUN_MODE}")
print(f"RUN_NAME: {run_name}")
print(f"SAVE: {RUN_DIR}")
print(f"EARLY_STOPPING: {USE_EARLY_STOPPING}, MIN_EPOCHS={MIN_EPOCHS_BEFORE_STOP}, PATIENCE={PATIENCE}")
print("=" * 80)

# 학습/검증 Dataset과 DataLoader 생성
train_dataset = SequenceDataset(
    DATA_ROOT,
    refresh_rate=REFRESH_RATE,
    seq_len=SEQ_LEN,
    split='train',
    val_ratio=VAL_RATIO,
    max_files_per_mood=MAX_FILES_PER_MOOD,
    samples_per_epoch=TRAIN_SAMPLES_PER_EPOCH,
    target_moods=active_moods,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

valid_dataset = SequenceDataset(
    DATA_ROOT,
    refresh_rate=REFRESH_RATE,
    seq_len=SEQ_LEN,
    split='valid',
    val_ratio=VAL_RATIO,
    max_files_per_mood=MAX_FILES_PER_MOOD,
    samples_per_epoch=VALID_SAMPLES_PER_EPOCH,
    target_moods=active_moods,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

# 모델, optimizer, checkpoint 재개 설정
model = CVAE_GRU(num_moods=11, hidden_dim=512, latent_dim=256).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)

early_stopping = EarlyStopping(
    patience=PATIENCE,
    min_epochs=MIN_EPOCHS_BEFORE_STOP,
    save_path=BEST_VALID_MODEL_PATH,
    enabled=USE_EARLY_STOPPING,
)

start_epoch = 0
best_loss_from_ckpt = None

if os.path.exists(LATEST_CHECKPOINT_PATH):
    print("🔄 latest checkpoint 발견. 중단 지점부터 이어서 학습합니다.")
    checkpoint = torch.load(LATEST_CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_loss_from_ckpt = checkpoint.get('best_valid_loss', None)
    early_stopping.best_loss = best_loss_from_ckpt
    early_stopping.best_epoch = checkpoint.get('best_epoch', None)

elif RUN_MODE == 'finetune' and LOAD_BASE_FOR_FINETUNE and os.path.exists(BASE_BEST_PATH):
    print(f"🔄 base best-valid 모델에서 finetune 시작: {BASE_BEST_PATH}")
    checkpoint = torch.load(BASE_BEST_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    # finetune에서는 optimizer state를 새로 시작한다.

elif os.path.exists(BEST_VALID_MODEL_PATH):
    print("🔄 기존 best-valid 모델 발견. 해당 모델부터 이어서 학습합니다.")
    checkpoint = torch.load(BEST_VALID_MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    if 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_loss_from_ckpt = checkpoint.get('valid_loss', checkpoint.get('loss', None))
    early_stopping.best_loss = best_loss_from_ckpt
    early_stopping.best_epoch = checkpoint.get('epoch', None)

# 처음 시작하거나 로그 파일이 없으면 CSV 헤더를 작성한다.
if start_epoch == 0 or not os.path.exists(LOSS_LOG_PATH):
    with open(LOSS_LOG_PATH, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            'Epoch', 'Beta',
            'TrainLoss', 'TrainBCE', 'TrainKLD',
            'ValidLoss', 'ValidBCE', 'ValidKLD',
            'ValidPredNotesPerChunk', 'ValidTargetNotesPerChunk',
            'ValidNotePrecision', 'ValidNoteRecall',
            'BestValidLoss'
        ])

# epoch 단위 학습과 검증 실행
total_train_steps = EPOCHS * len(train_loader)
beta_anneal_steps = max(1, int(total_train_steps * BETA_ANNEAL_RATIO))
global_step = start_epoch * len(train_loader)

train_loss_history = []
valid_loss_history = []
start_epoch_for_plot = start_epoch

print("🔥 Final train 시작")
print("   - train loss는 학습 추세 확인")
print("   - valid loss는 일반화 감시")
print("   - note count / precision / recall은 '0만 맞히는 모델' 감시")
print("   - finetune에서는 checkpoint를 직접 생성해 들어보고 선택")

for epoch in range(start_epoch, EPOCHS):
    # 전체 학습 초반부 동안 beta를 0에서 1까지 서서히 올린다.
    global_step = epoch * len(train_loader)
    beta_val = min(1.0, global_step / beta_anneal_steps)

    train_metrics = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        beta_val=beta_val,
        note_threshold=NOTE_THRESHOLD,
        track_note=False,  # train note metric은 시간 절약을 위해 생략
    )

    valid_metrics = run_epoch(
        model,
        valid_loader,
        optimizer=None,
        beta_val=beta_val,
        note_threshold=NOTE_THRESHOLD,
        track_note=True,
    )

    avg_train_loss = train_metrics['loss']
    avg_valid_loss = valid_metrics['loss']

    train_loss_history.append(avg_train_loss)
    valid_loss_history.append(avg_valid_loss)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {train_metrics['loss']:.4f} | BCE: {train_metrics['bce']:.4f} | KLD: {train_metrics['kld']:.4f} || "
        f"Valid Loss: {valid_metrics['loss']:.4f} | BCE: {valid_metrics['bce']:.4f} | KLD: {valid_metrics['kld']:.4f} || "
        f"Valid Notes pred/target: {valid_metrics['pred_notes_per_chunk']:.1f}/{valid_metrics['target_notes_per_chunk']:.1f} | "
        f"P/R: {valid_metrics['note_precision']:.4f}/{valid_metrics['note_recall']:.4f} | "
        f"Beta: {beta_val:.4f}"
    )

    # 중단 후 재개할 수 있도록 latest checkpoint를 저장한다.
    latest_payload = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_metrics['loss'],
        'valid_loss': valid_metrics['loss'],
        'best_valid_loss': early_stopping.best_loss,
        'best_epoch': early_stopping.best_epoch,
        'beta': beta_val,
        'run_mode': RUN_MODE,
        'target_moods': active_moods,
    }
    torch.save(latest_payload, LATEST_CHECKPOINT_PATH)

    # valid loss 기준 best checkpoint와 early stopping 상태를 갱신한다.
    early_stopping(
        avg_valid_loss,
        model,
        optimizer,
        epoch,
        extra={
            'train_loss': train_metrics['loss'],
            'beta': beta_val,
            'run_mode': RUN_MODE,
            'target_moods': active_moods,
        }
    )

    # 지정된 epoch마다 생성 비교용 후보 checkpoint를 저장한다.
    should_save_candidate = ((epoch + 1) % CHECKPOINT_EVERY == 0) or ((epoch + 1) in EXTRA_SAVE_EPOCHS)
    if should_save_candidate:
        candidate_path = os.path.join(CKPT_DIR, f'epoch_{epoch+1:03d}.pth')
        torch.save(latest_payload, candidate_path)
        print(f"🎧 후보 checkpoint 저장: {candidate_path}")

    with open(LOSS_LOG_PATH, 'a', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            epoch + 1,
            f"{beta_val:.6f}",
            f"{train_metrics['loss']:.6f}",
            f"{train_metrics['bce']:.6f}",
            f"{train_metrics['kld']:.6f}",
            f"{valid_metrics['loss']:.6f}",
            f"{valid_metrics['bce']:.6f}",
            f"{valid_metrics['kld']:.6f}",
            f"{valid_metrics['pred_notes_per_chunk']:.6f}",
            f"{valid_metrics['target_notes_per_chunk']:.6f}",
            f"{valid_metrics['note_precision']:.6f}",
            f"{valid_metrics['note_recall']:.6f}",
            '' if early_stopping.best_loss is None else f"{early_stopping.best_loss:.6f}",
        ])

    if early_stopping.early_stop:
        print(
            f"🚨 Valid loss 기준 조기 종료. "
            f"best epoch={early_stopping.best_epoch + 1 if early_stopping.best_epoch is not None else 'unknown'}, "
            f"best valid={early_stopping.best_loss:.4f}"
        )
        break

# 학습이 진행된 경우 train/valid loss 그래프를 저장한다.
if len(train_loss_history) > 0:
    print("\n📊 학습 완료. Loss 그래프를 저장합니다.")
    plt.figure(figsize=(10, 5))
    x_epochs = range(start_epoch_for_plot + 1, start_epoch_for_plot + 1 + len(train_loss_history))
    plt.plot(x_epochs, train_loss_history, marker='o', linestyle='-', label='Train Loss')
    plt.plot(x_epochs, valid_loss_history, marker='o', linestyle='-', label='Valid Loss')
    plt.title(f'Train / Valid Loss over Epochs - {run_name}')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.savefig(GRAPH_PATH)
    plt.show()

print("\n✅ 완료")
print(f"best-valid model: {BEST_VALID_MODEL_PATH}")
print(f"latest checkpoint: {LATEST_CHECKPOINT_PATH}")
print(f"candidate checkpoints: {CKPT_DIR}")
print(f"loss log: {LOSS_LOG_PATH}")
print(f"loss graph: {GRAPH_PATH}")
